# A Dash application

A dashboard is a chart plus controls plus a callback that redraws it. This notebook assembles the
Gapminder dashboard defined in [`../app/dashboard.py`](../app/dashboard.py), inspects its layout,
and exercises the callback logic directly — without starting a web server.

## Learning objectives

By the end of this notebook you will be able to:

- describe the component tree of a Dash layout;
- connect inputs to an output with a callback;
- keep data filtering and figure building in plain testable functions;
- test a callback's logic without launching the server;
- run the app as a standalone process.

## Concept

Dash apps have three parts. The **layout** is a tree of components (`html.Div`, `dcc.Graph`,
`dcc.Slider`). **Callbacks** declare that when an input changes, an output is recomputed; the
`@app.callback(Output(...), Input(...))` decorator wires them together. The **server** serves the
layout and marshals callback requests. Keeping the heavy lifting in ordinary functions
(`filter_data`, `bubble_figure`) means the logic is importable and testable without a browser.

An interactive app differs from a static notebook in that the reader chooses the view. That is
powerful but also a responsibility: every control needs a sensible default, and every callback
must handle the values the controls can produce.

## Worked example

### Import the app module and load data

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
sys.path.insert(0, str(Path.cwd().parent / "app"))
import dashboard
from ds_practice import load_gapminder, set_seed

set_seed(42)
gap = load_gapminder()
print("rows:", len(gap), "| years:", gap["year"].min(), "-", gap["year"].max())

### Filtering logic

`filter_data` is a pure function: give it a continent and a year and it returns a subset. That is
what makes it easy to test.

In [ ]:
subset = dashboard.filter_data(gap, continent="Europe", year=2007)
print("Europe 2007 rows:", len(subset))
print("continents in full data:", sorted(gap["continent"].unique()))

### The figure builder

`bubble_figure` turns a filtered frame into a Plotly figure. Its output is data, not a picture, so
we can inspect its traces.

In [ ]:
figure = dashboard.bubble_figure(subset)
print("traces:", len(figure.data))
print("title:", figure.layout.title.text)
figure.show()

### The layout

`create_app` builds the component tree. We can read the children and the ids that callbacks refer
to, which is the quickest way to understand an unfamiliar app.

In [ ]:
app = dashboard.create_app(gap)
children = app.layout.children
print("top-level children:", [type(c).__name__ for c in children])

graph_ids = [c.id for c in children if getattr(c, "id", None) in {"bubble", "continent", "year"}]
print("control ids found:", graph_ids)

### The callback

The callback is registered on the app. Calling its function with explicit inputs reproduces what
the browser would do, without a server.

In [ ]:
filtered = dashboard.filter_data(gap, continent="Africa", year=1987)
print("African countries in 1987:", len(filtered))
figure = dashboard.bubble_figure(filtered)
figure.show()

### Running the app

From a terminal, the app starts a local server:

```bash
python 08-data-visualization/app/dashboard.py   # http://127.0.0.1:8050
```

The notebook deliberately does not start it, because a blocking server would stop the rest of the
notebook from running.

## Exercises

1. **Add a control.** Sketch (in prose and code) a second dropdown that filters by country, and
   state which component would be the `Output` and which the new `Input`.
2. **Test the pure functions.** Write assertions for `filter_data` with `continent="All"` and with
   a year that has no rows, and explain the expected behaviour of each.
3. **Default view.** Argue for a default continent and year, then check how many countries the
   default view shows and whether that is a good first impression.

## Limitations

A Dash app running in a notebook shares the kernel, so a blocking `app.run_server()` would freeze
it; the app is meant to be launched in a terminal. The callback here is simple and stateless, but
real apps face concurrency, caching, and authentication. The layout is rebuilt on every callback
in this design, which is fine for one graph but wasteful for many. Finally, a dashboard is only as
good as its data freshness: the Gapminder series ends in 2007 and does not update.